# DESC ELAsTiCC2 — SALT2 fit with free redshift and sncosmo

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-09
- **last update** : 2026-05-11 : add error on distance modulus
- **last update**: 2026-05-12 : refactor computemustaterrors (gradient g^T.C.g)
- **based on** : `02_sncosmo/02_elasticc2_fitsalt2_lightcurves.ipynb` (z fixed)
- **inspired by** : `02_sncosmo/03b_elasticc2_fitsalt3andredshift_lightcurves.ipynb` (SALT3 free-z)

## Purpose

In the real observational scenario, the redshift of a transient is **not known a priori**
(except when a spectroscopic host redshift is available, which is not guaranteed for Rubin LSST).
This notebook therefore fits the **five SALT2 parameters simultaneously**:

$$
\text{free params} = \{z,\; t_0,\; x_0,\; x_1,\; c\}
$$

The truth redshift `ZCMB` from the ELAsTiCC2 catalogue is **only used**:
1. as the reference for validation at the end of the notebook,
2. **not** used to constrain the fit bounds (we apply fully agnostic survey-level bounds).

## Redshift fitting strategy

Fitting redshift from broadband light curves alone is notoriously degenerate:
the colour `c` and the redshift `z` both shift spectral features,
and the amplitude `x0` absorbs the luminosity-distance dependence.
A single local minimisation starting from a poor initial guess
frequently lands in a wrong local minimum.

We use a **two-pass strategy** identical to that of notebook `03b`:

1. **Coarse grid scan** — for a grid of `N_GRID_Z` redshift values spanning
   `[Z_BOUND_LO, Z_BOUND_HI]`, fix `z` and run a fast 4-parameter fit
   `(t0, x0, x1, c)`.  Record the χ².
2. **Refined free-z fit** — initialise at the `z` value that gave the lowest χ²
   in the grid scan, then run `sncosmo.fit_lc` with all 5 parameters free
   within a narrow window around that starting point.

## Final diagnostic figure

The last section produces an **errorbar plot** of `z_fit ± σ_z` versus `z_true`
with a reference diagonal, colour-coded by χ²/dof, plus residual panels
showing $\Delta z$ and $\Delta z / (1 + z_{\rm true})$.

### References
- Guy et al. 2007 — SALT2: https://doi.org/10.1051/0004-6361:20066930
- sncosmo documentation: https://sncosmo.readthedocs.io/en/stable/index.html
- ELAsTiCC2 dataset: DESC TD public data
- Notebook `02_sncosmo/02_elasticc2_fitsalt2_lightcurves.ipynb` — z-fixed SALT2 baseline
- Notebook `02_sncosmo/03b_elasticc2_fitsalt3andredshift_lightcurves.ipynb` — SALT3 free-z strategy


## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging
import warnings

import numpy as np
import pandas as pd
import astropy.table
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.gridspec import GridSpec

import sncosmo
from astropy.cosmology import FlatLambdaCDM

# ── local library ──────────────────────────────────────────────────────────────
libdir = pathlib.Path(os.getcwd()).parent.parent / "lib_elasticc2"
sys.path.insert(0, str(libdir))
from transcientslightcurves import elasticc2_snana_reader

# ── logging ────────────────────────────────────────────────────────────────────
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(
        logging.Formatter("[%(asctime)s - %(levelname)s] - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    )
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 1 · Parameters

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy
ZPAB = 2.5 * np.log10(AB_FLUX_ZERO)  # mag  ≈ 31.4

In [ ]:
# ── Data selection ─────────────────────────────────────────────────────────────
OBJ_CLASS = "SNIa-SALT3"  # SNANA class label used for the ELAsTiCC2 sample
Z_MIN = 0.001  # truth-redshift lower bound for sample selection
Z_MAX = 1.5  # truth-redshift upper bound
FILE_NUM = 1  # PHOT file index (1–40; None = all)
MIN_DETECTIONS = 8  # minimum detected points per object
DETECTED_ONLY = True  # use only detected points (PHOTFLAG & photflag_detect)
N_CURVES = 1000  # number of events to fit
RANDOM_SEED = 42

# ── SALT2 model ────────────────────────────────────────────────────────────────
SALT2_SOURCE = "salt2-extended"  # sncosmo source name
ZP = ZPAB  # zero-point (AB system)
ZPSYS = "ab"
BANDS = ["u", "g", "r", "i", "z", "y"]
BAND_PREFIX = "lsst"

# ── Redshift fit bounds ────────────────────────────────────────────────────────
# Applied to the z fit without using the truth redshift.
# SALT2-extended covers rest-frame 2000–9200 Å;
# the bluest LSST band (u, λ_eff ≈ 3700 Å) gives a practical lower limit ~0.02.
Z_BOUND_LO = 0.02  # hard lower bound for z in the fit
Z_BOUND_HI = 1.20  # hard upper bound for z in the fit

# ── Two-pass strategy parameters ──────────────────────────────────────────────
N_GRID_Z = 30  # number of z grid points in the coarse scan
Z_REFINE_HALF = 0.15  # half-width of the z search window for the refined fit

# ── Data path ──────────────────────────────────────────────────────────────────
DATA_DIR = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX = "ELASTICC2_TRAIN_02_"

# ── Plot colours per band ──────────────────────────────────────────────────────
BAND_COLORS = {"u": "#cc0ccc", "g": "#00cc44", "r": "#cc0000", "i": "#ff4400", "z": "#886600", "y": "#442200"}
NCOLS = 4

rng = np.random.default_rng(seed=RANDOM_SEED)
print(f"SALT2 source : {SALT2_SOURCE}")
print(f"ZP={ZP:.3f}  zpsys={ZPSYS}")
print(f"Redshift fit : z ∈ [{Z_BOUND_LO}, {Z_BOUND_HI}]  (no truth used in fit)")
print(f"Grid scan    : {N_GRID_Z} points  |  refine window: ±{Z_REFINE_HALF}")
print(f"N_CURVES     : {N_CURVES}")

## 2 · SALT2 model and band registration check

In [ ]:
# Load SALT2 model — sncosmo downloads it on first use
salt2_model = sncosmo.Model(source=SALT2_SOURCE)
print(f"Model       : {salt2_model.source.name}")
print(f"Parameters  : {salt2_model.param_names}")
print(f"Phase range : {salt2_model.source.minphase():.1f} – {salt2_model.source.maxphase():.1f} days")
print(f"Wave range  : {salt2_model.source.minwave():.0f} – {salt2_model.source.maxwave():.0f} Å")
print(f"\nFit bounds: z ∈ [{Z_BOUND_LO}, {Z_BOUND_HI}]")

print("\nLSST band registration:")
for b in BANDS:
    bp = sncosmo.get_bandpass(BAND_PREFIX + b)
    print(f"  lsst{b}  λ_eff={bp.wave_eff:.0f} Å  ✓")

## 3 · Load ELAsTiCC2 data

In [ ]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)

_logger.info(f"Loading {OBJ_CLASS}...")
head = esr.get_head(OBJ_CLASS, return_format="pandas")
truth = esr.get_object_truth(OBJ_CLASS, return_format="pandas")
all_ltcvs = esr.get_all_ltcvs(OBJ_CLASS, file_num=FILE_NUM, return_format="pandas")
_logger.info("Done.")

print(f"{all_ltcvs['SNID'].nunique()} objects loaded.")
print(f"Columns: {list(all_ltcvs.columns)}")

In [ ]:
# ── Filter: redshift range + minimum detections ───────────────────────────────
detcounts = (
    all_ltcvs[(all_ltcvs["PHOTFLAG"] & esr.photflag_detect) != 0]
    .groupby("SNID")
    .agg("count")["MJD"]
    .reset_index()
    .rename({"MJD": "ndetect"}, axis=1)
)
truth_counts = truth.join(detcounts.set_index("SNID"), on="SNID", how="inner")
subset = truth_counts[
    (truth_counts["ZCMB"] >= Z_MIN)
    & (truth_counts["ZCMB"] < Z_MAX)
    & (truth_counts["ndetect"] >= MIN_DETECTIONS)
].copy()
print(f"{len(subset)} objects pass selection (z∈[{Z_MIN},{Z_MAX}), ndet≥{MIN_DETECTIONS}).")

## 4 · Helper functions

### 4.1 · ELAsTiCC2 → sncosmo table

In [ ]:
def make_sncosmo_table(
    ltcv_df, detected_only=True, photflag_detect=None, zp=ZP, zpsys=ZPSYS, band_prefix=BAND_PREFIX
):
    """Convert an ELAsTiCC2 light-curve DataFrame to an astropy.Table
    suitable for sncosmo.fit_lc.

    Parameters
    ----------
    ltcv_df         : DataFrame for a single SNID
    detected_only   : keep only rows where PHOTFLAG & photflag_detect != 0
    photflag_detect : bitmask value for detections
    zp              : photometric zero-point
    zpsys           : zero-point system ('ab')
    band_prefix     : prefix to prepend to the band letter ('lsst')

    Returns
    -------
    astropy.Table with columns: time, band, flux, fluxerr, zp, zpsys
    """
    df = ltcv_df.copy()
    if detected_only and photflag_detect is not None:
        df = df[(df["PHOTFLAG"] & photflag_detect) != 0]
    df = df[df["FLUXCALERR"] > 0].copy()
    df["BAND"] = df["BAND"].str.strip().str.lower()
    df["band_sncosmo"] = band_prefix + df["BAND"]
    return astropy.table.Table(
        {
            "time": df["MJD"].values.astype(float),
            "band": df["band_sncosmo"].values,
            "flux": df["FLUXCAL"].values.astype(float),
            "fluxerr": df["FLUXCALERR"].values.astype(float),
            "zp": np.full(len(df), zp, dtype=float),
            "zpsys": np.full(len(df), zpsys),
        }
    )


print("make_sncosmo_table ready.")

### 4.2 · Two-pass SALT2 fitter with free redshift

**Pass 1 — coarse grid scan**  
For each `z` in a uniform grid over `[Z_BOUND_LO, Z_BOUND_HI]`, the SALT2 model is
initialised at that `z` (fixed) and a fast 4-parameter fit `(t0, x0, x1, c)` is run.
The grid point with the lowest χ² becomes the starting point for pass 2.

**Pass 2 — refined free-z fit**  
All 5 parameters `(z, t0, x0, x1, c)` are freed, with `z` bounded within
`[z_best − Z_REFINE_HALF, z_best + Z_REFINE_HALF]` (clipped to global bounds).
The covariance matrix from this fit provides `σ_z` and other uncertainties.

**Comparison with SALT3 (notebook 03b)**  
The fitting strategy is identical; only the sncosmo source changes from `salt3` to
`salt2-extended`. This allows a direct comparison of the photometric redshift
accuracy of the two SALT-family templates on the same ELAsTiCC2 sample.

In [ ]:
def fit_salt2_free_z(ltcv_df, photflag_detect=None, z_init_hint=None) -> dict:
    """Fit a single ELAsTiCC2 event with SALT2, treating z as a free parameter.

    Uses a two-pass strategy:
      Pass 1 — coarse grid scan over z → best starting z_grid
      Pass 2 — refined 5-parameter free fit around z_grid

    Parameters
    ----------
    ltcv_df         : DataFrame for one SNID
    photflag_detect : bitmask for detections (from esr.photflag_detect)
    z_init_hint     : optional initial z hint (e.g. host photo-z);
                      if provided, the grid is centred on this value instead
                      of spanning the full [Z_BOUND_LO, Z_BOUND_HI] range

    Returns
    -------
    dict with keys: success, z, z_err, t0, x0, x1, c, chi2, ndof, chi2_red,
                    vcov, param_errors, z_grid, chi2_grid, z_grid_best,
                    fitted_model, table, message
    """
    # ── Build observation table ────────────────────────────────────────────────
    try:
        obs = make_sncosmo_table(ltcv_df, detected_only=DETECTED_ONLY, photflag_detect=photflag_detect)
    except Exception as e:
        return {"success": False, "message": f"Table error: {e}"}
    if len(obs) < 6:
        return {"success": False, "message": "Too few data points"}

    t0_guess = float(obs["time"][np.argmax(obs["flux"])])

    # ── Define coarse z grid ──────────────────────────────────────────────────
    if z_init_hint is not None:
        z_lo = max(Z_BOUND_LO, z_init_hint - Z_REFINE_HALF)
        z_hi = min(Z_BOUND_HI, z_init_hint + Z_REFINE_HALF)
    else:
        z_lo = Z_BOUND_LO
        z_hi = Z_BOUND_HI
    z_grid = np.linspace(z_lo, z_hi, N_GRID_Z)

    # ── Pass 1: coarse scan (z fixed, 4 free params) ──────────────────────────
    chi2_grid = np.full(N_GRID_Z, np.inf)
    bounds_4par = {
        "t0": (t0_guess - 30.0, t0_guess + 30.0),
        "x0": (1e-8, 1.0),
        "x1": (-5.0, 5.0),
        "c": (-0.5, 0.5),
    }

    for k, z_try in enumerate(z_grid):
        m = sncosmo.Model(source=SALT2_SOURCE)
        m.set(z=z_try, t0=t0_guess, x0=1e-4, x1=0.0, c=0.0)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res_k, _ = sncosmo.fit_lc(
                    obs, m, vparam_names=["t0", "x0", "x1", "c"], bounds=bounds_4par, minsnr=0.0, warn=False
                )
            chi2_grid[k] = res_k.chisq
        except Exception:
            pass

    best_k = int(np.argmin(chi2_grid))
    z_grid_best = float(z_grid[best_k])

    # ── Pass 2: refined free-z fit (5 free params) ────────────────────────────
    z_ref_lo = max(Z_BOUND_LO, z_grid_best - Z_REFINE_HALF)
    z_ref_hi = min(Z_BOUND_HI, z_grid_best + Z_REFINE_HALF)

    model_final = sncosmo.Model(source=SALT2_SOURCE)
    model_final.set(z=z_grid_best, t0=t0_guess, x0=1e-4, x1=0.0, c=0.0)

    bounds_5par = {
        "z": (z_ref_lo, z_ref_hi),
        "t0": (t0_guess - 30.0, t0_guess + 30.0),
        "x0": (1e-8, 1.0),
        "x1": (-5.0, 5.0),
        "c": (-0.5, 0.5),
    }

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result, fitted_model = sncosmo.fit_lc(
                obs,
                model_final,
                vparam_names=["z", "t0", "x0", "x1", "c"],
                bounds=bounds_5par,
                minsnr=0.0,
                warn=False,
            )
    except Exception as e:
        return {
            "success": False,
            "message": f"Pass-2 fit failed: {e}",
            "z_grid": z_grid,
            "chi2_grid": chi2_grid,
            "z_grid_best": z_grid_best,
            "table": obs,
        }

    chi2 = float(result.chisq)
    ndof = int(result.ndof)
    chi2_red = chi2 / max(ndof, 1)

    # ── Covariance and 1-sigma errors ─────────────────────────────────────────
    vparam_names = ["z", "t0", "x0", "x1", "c"]
    vcov = result.covariance
    param_errors = {}
    if vcov is not None:
        diag = np.diag(vcov)
        for i, p in enumerate(vparam_names):
            param_errors[p] = float(np.sqrt(max(diag[i], 0.0)))

    return {
        "success": True,
        "result": result,
        "fitted_model": fitted_model,
        "table": obs,
        "vparam_names": vparam_names,
        "z": float(fitted_model["z"]),
        "z_err": param_errors.get("z", float("nan")),
        "t0": float(fitted_model["t0"]),
        "x0": float(fitted_model["x0"]),
        "x1": float(fitted_model["x1"]),
        "c": float(fitted_model["c"]),
        "chi2": chi2,
        "ndof": ndof,
        "chi2_red": chi2_red,
        "vcov": vcov,
        "param_errors": param_errors,
        "z_grid": z_grid,
        "chi2_grid": chi2_grid,
        "z_grid_best": z_grid_best,
        "message": result.message,
    }


print("Two-pass SALT2 free-z fitter ready.")

## 5 · Select events and run fits

In [ ]:
n_avail = min(N_CURVES, len(subset))
chosen_idx = rng.choice(len(subset), size=n_avail, replace=False)
chosen_snids = subset["SNID"].values[chosen_idx]
print(f"Selected {n_avail} SNIDs for free-z SALT2 fitting.")

In [ ]:
def computemustaterrors(
    param_names, params, covariance, mb_const=10.635, M=-19.3, alpha=0.14, beta=3.1, cosmo=None
):
    """Propagate SALT2 fit covariance into an uncertainty on the Tripp distance modulus.

    Tripp formula
    -------------
        mu  =  m_B*  -  M  +  alpha*x1  -  beta*c
        m_B*  =  -2.5 * log10(x0)  +  mb_const

    Partial derivatives of mu (gradient vector g):
        d(mu)/d(x0) = -2.5 / (x0 * ln10)
        d(mu)/d(x1) =  alpha
        d(mu)/d(c)  = -beta
        d(mu)/d(z)  = (5/ln10)*(1/d_L)*dd_L/dz   [only when z is a free parameter]

    The full error propagation is evaluated as:
        sigma_mu = sqrt( g^T . C_sub . g )
    where C_sub is the covariance sub-matrix restricted to the parameters
    that enter mu (x0, x1, c and optionally z).  t0 is excluded.

    Parameters
    ----------
    param_names : list[str]
        Names of *all* fitted parameters, in the same order as `params` and
        the rows/columns of `covariance`.
        z fixed : ['t0', 'x0', 'x1', 'c']
        z free  : ['z', 't0', 'x0', 'x1', 'c']
    params      : array-like  – fitted values, same order as param_names.
    covariance  : 2-D array   – full covariance matrix, same order.
    mb_const    : SALT2 zero-point offset (default 10.635 for salt2-extended).
    M           : absolute B-band magnitude of a standard SNIa.
    alpha       : Tripp stretch coefficient.
    beta        : Tripp colour coefficient.
    cosmo       : astropy cosmology instance; used only when z is free.
                  Defaults to FlatLambdaCDM(H0=70, Om0=0.3).

    Returns
    -------
    dict with keys:
        mu          – Tripp distance modulus
        sigma_mu    – 1-sigma uncertainty on mu
        gradient    – {param_name: d(mu)/d(param)} for mu-entering params
        sigmas      – {param_name: 1-sigma marginal error} for all params
        covariances – {'pi:pj': cov(pi,pj)} for mu-entering pairs
    """
    params = np.asarray(params, dtype=float)
    cov = np.asarray(covariance, dtype=float)
    names = list(param_names)

    # Build a name→index map so we never hard-code positions
    idx = {name: i for i, name in enumerate(names)}

    # Retrieve the three SALT2 photometric parameters by name
    x0 = params[idx["x0"]]
    x1 = params[idx["x1"]]
    c = params[idx["c"]]

    # Tripp distance modulus
    mB_star = -2.5 * np.log10(x0) + mb_const
    mu = mB_star - M + alpha * x1 - beta * c

    # Gradient of mu w.r.t. the parameters that enter it
    gradient = {
        "x0": -2.5 / (x0 * np.log(10)),  # d(m_B*)/d(x0)
        "x1": alpha,  # d(mu)/d(x1)
        "c": -beta,  # d(mu)/d(c)
    }
    mu_params = ["x0", "x1", "c"]

    # Add the z contribution only when z was a free parameter
    if "z" in idx:
        z = params[idx["z"]]
        if cosmo is None:
            from astropy.cosmology import FlatLambdaCDM

            cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
        dz = 1e-5
        dL = cosmo.luminosity_distance(z).value
        dL_p = cosmo.luminosity_distance(z + dz).value
        dL_m = cosmo.luminosity_distance(z - dz).value
        gradient["z"] = (5.0 / np.log(10)) * (dL_p - dL_m) / (2.0 * dz * dL)
        mu_params.append("z")

    # Gradient vector g and covariance sub-matrix C_sub
    # restricted to the parameters that actually enter mu  (t0 excluded)
    g = np.array([gradient[p] for p in mu_params])
    sub_idx = np.array([idx[p] for p in mu_params])
    C_sub = cov[np.ix_(sub_idx, sub_idx)]

    # sigma_mu^2 = g^T . C_sub . g  (exact first-order error propagation)
    sigma_mu = float(np.sqrt(max(0.0, g @ C_sub @ g)))

    # Marginal 1-sigma errors on every fitted parameter
    sigmas = {name: float(np.sqrt(max(0.0, cov[i, i]))) for name, i in idx.items()}

    # Off-diagonal covariances for mu-entering pairs (upper triangle only)
    covariances = {}
    for ia, a in enumerate(mu_params):
        for ib, b in enumerate(mu_params):
            if ia < ib:
                covariances[f"{a}:{b}"] = float(C_sub[ia, ib])

    return {
        "mu": float(mu),
        "sigma_mu": sigma_mu,
        "gradient": gradient,
        "sigmas": sigmas,
        "covariances": covariances,
    }


print("computemustaterrors ready  (gradient g^T.C.g, param_names-aware)")

In [ ]:
fit_results = {}
fit_errors = {}

for snid in chosen_snids:
    ltcv = all_ltcvs[all_ltcvs["SNID"] == snid]
    z_row = truth[truth["SNID"] == snid]["ZCMB"].values
    z_true = float(z_row[0]) if len(z_row) else np.nan

    # Truth z is NOT passed to the fitter — only used here for display / validation
    res = fit_salt2_free_z(ltcv, photflag_detect=esr.photflag_detect)
    fit_results[snid] = res

    # extract parameters and covariance
    param_names = res["result"].param_names
    params = res["result"].parameters
    cov = res["result"].covariance

    fit_errors[snid] = computemustaterrors(param_names, params, cov)

    # param_names = res["result"].param_names
    param_names = res["result"].vparam_names
    params = res["result"].parameters
    cov = res["result"].covariance
    fit_errors[snid] = computemustaterrors(param_names, params, cov)
    sigma_mu = fit_errors[snid]["sigma_mu"]

    if res["success"]:
        pe = res["param_errors"]
        dz = res["z"] - z_true
        print(
            f"  SNID {snid:8d}  ✓  "
            f"z_true={z_true:.4f}  z_fit={res['z']:.4f}±{pe.get('z', float('nan')):.4f}  "
            f"Δz={dz:+.4f}  "
            f"x1={res['x1']:+.3f}  c={res['c']:+.3f}  "
            f"χ²/dof={res['chi2_red']:.2f}"
            f"σ_mu= {sigma_mu:.2f}"
        )
    else:
        print(f"  SNID {snid:8d}  ✗  {res['message']}")

## 6 · Per-event plot: light curve + χ²(z) grid scan

For each event two panels are shown side by side:
- **Left**: multi-band light curve with fitted SALT2 model.
- **Right**: χ²(z) from the coarse grid scan; the best-z from the grid (Pass 1) and
  the final fitted z (Pass 2) are marked, with the truth z shown for reference.

In [ ]:
def plot_event_with_zgrid(snid, res, z_true):
    """Two-panel figure: light curve (left) + χ²(z) grid scan (right)."""
    fig, (ax_lc, ax_z) = plt.subplots(
        1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [2, 1]}, tight_layout=True
    )

    # ── Left panel: light curve ────────────────────────────────────────────────
    if res.get("success"):
        obs = res["table"]
        fitted_model = res["fitted_model"]
        t0 = res["t0"]
        pe = res["param_errors"]
        t_dense = np.linspace(float(obs["time"].min()) - 10, float(obs["time"].max()) + 10, 400)
        for b in BANDS:
            bname = BAND_PREFIX + b
            color = BAND_COLORS[b]
            mask = np.array(obs["band"]) == bname
            if mask.sum() > 0:
                ax_lc.errorbar(
                    obs["time"][mask] - t0,
                    obs["flux"][mask],
                    yerr=obs["fluxerr"][mask],
                    color=color,
                    ls="None",
                    marker="o",
                    ms=4,
                    capsize=2,
                    label=b,
                    zorder=3,
                )
            try:
                f_model = fitted_model.bandflux(bname, t_dense, zp=ZP, zpsys=ZPSYS)
                valid = np.isfinite(f_model)
                if valid.sum() > 1:
                    ax_lc.plot(t_dense[valid] - t0, f_model[valid], color=color, lw=1.5, zorder=2)
            except Exception:
                pass
        ax_lc.axhline(0.0, color="k", lw=0.5, ls="--")
        ax_lc.set_title(
            f"SNID {snid}  z_true={z_true:.4f}\n"
            f"z_fit={res['z']:.4f}±{pe.get('z', float('nan')):.4f}  "
            f"x1={res['x1']:+.2f}  c={res['c']:+.2f}  "
            f"χ²/dof={res['chi2_red']:.2f}",
            fontsize=8,
        )
        ax_lc.legend(fontsize=6, ncol=6, loc="upper right")
    else:
        ax_lc.set_title(f"SNID {snid}\nFit failed", color="red", fontsize=9)

    ax_lc.set_xlabel(r"$t - t_0$ [days]", fontsize=9)
    ax_lc.set_ylabel("FLUXCAL", fontsize=9)

    # ── Right panel: χ²(z) grid scan ─────────────────────────────────────────
    z_grid = res.get("z_grid")
    chi2_grid = res.get("chi2_grid")
    if z_grid is not None and chi2_grid is not None:
        finite = np.isfinite(chi2_grid)
        ax_z.plot(z_grid[finite], chi2_grid[finite], "C0o-", ms=4, lw=1.2, label="grid χ²")

        z_best_grid = res.get("z_grid_best", float("nan"))
        ax_z.axvline(z_best_grid, color="C0", ls="--", lw=1.2, label=f"grid best: {z_best_grid:.4f}")

        if res.get("success"):
            ax_z.axvline(res["z"], color="C1", ls="-", lw=1.5, label=f"z_fit: {res['z']:.4f}")

        ax_z.axvline(z_true, color="k", ls=":", lw=1.5, label=f"z_true: {z_true:.4f}")

        # Shade refine window
        lo = max(Z_BOUND_LO, z_best_grid - Z_REFINE_HALF)
        hi = min(Z_BOUND_HI, z_best_grid + Z_REFINE_HALF)
        ax_z.axvspan(lo, hi, alpha=0.12, color="C0", label="refine window")

        chi2_min = chi2_grid[finite].min() if finite.any() else 0
        ax_z.set_ylim(
            chi2_min * 0.9, min(chi2_grid[finite].max(), chi2_min * 3 + 10) if finite.any() else 100
        )
        ax_z.legend(fontsize=7, loc="upper right")

    ax_z.set_xlabel("Redshift z", fontsize=9)
    ax_z.set_ylabel(r"$\chi^2$ (Pass-1, fixed z)", fontsize=9)
    ax_z.set_title("Coarse grid scan", fontsize=9)

    return fig


# ── Draw per-event figures (limit to first N_PLOT events) ──────────────────────
N_PLOT = min(n_avail, 20)

for snid in chosen_snids[:N_PLOT]:
    z_row = truth[truth["SNID"] == snid]["ZCMB"].values
    z_true = float(z_row[0]) if len(z_row) else np.nan
    fig = plot_event_with_zgrid(snid, fit_results[snid], z_true)
    plt.show()

## 7 · Summary table

**Bug fix note**: `x0` is included in `row.update()` so that the Hubble diagram
in section 9 (which needs `df_ok['x0']`) works correctly.

In [ ]:
rows = []
for snid in chosen_snids:
    res = fit_results[snid]
    z_row = truth[truth["SNID"] == snid]["ZCMB"].values
    z_true = float(z_row[0]) if len(z_row) else np.nan
    pe = res.get("param_errors", {})

    sigma_mu = fit_errors[snid]["sigma_mu"]

    row = {"SNID": snid, "z_true": round(z_true, 4), "success": res.get("success", False)}
    if res.get("success"):
        dz = res["z"] - z_true
        row.update(
            {
                "z_fit": round(res["z"], 4),
                "z_err": round(pe.get("z", float("nan")), 4),
                "dz": round(dz, 4),
                "dz_1pz": round(dz / (1.0 + z_true), 4),
                "x0": float(f"{res['x0']:.4e}"),  # needed for Hubble diagram
                "x1": round(res["x1"], 3),
                "x1_err": round(pe.get("x1", float("nan")), 3),
                "c": round(res["c"], 3),
                "c_err": round(pe.get("c", float("nan")), 3),
                "chi2": round(res["chi2"], 2),
                "ndof": res["ndof"],
                "chi2_red": round(res["chi2_red"], 3),
                "sigma_mu": round(sigma_mu, 2),
            }
        )
    rows.append(row)

df_results = pd.DataFrame(rows)
df_ok = df_results[df_results["success"]].copy()

print(f"Fit summary  : {len(df_ok)} / {n_avail} converged")
print(f"Median Δz       : {df_ok['dz'].median():+.4f}")
print(f"RMS Δz          : {df_ok['dz'].std():.4f}")
print(f"Median Δz/(1+z) : {df_ok['dz_1pz'].median():+.5f}")
print(f"RMS  Δz/(1+z)   : {df_ok['dz_1pz'].std():.5f}")
print()
print(df_ok[["SNID", "z_true", "z_fit", "z_err", "dz", "dz_1pz", "chi2_red"]].to_string(index=False))

## 8 · Distribution of SALT2 parameters

In [ ]:
params = ["x1", "c", "chi2_red", "x1_err", "c_err"]
labels = {
    "x1": r"SALT2 $x_1$ (stretch)",
    "c": r"SALT2 $c$ (colour)",
    "chi2_red": r"$\chi^2$/dof",
    "x1_err": r"$\sigma(x_1)$",
    "c_err": r"$\sigma(c)$",
}
fig2, axes2 = plt.subplots(1, len(params), figsize=(4.5 * len(params), 3.5), tight_layout=True)
for ax, par, col in zip(axes2, params, ["C0", "C1", "C2", "C3", "C4"]):
    vals = df_ok[par].dropna()
    if len(vals) == 0:
        continue
    ax.hist(vals, bins=12, color=col, edgecolor="white")
    ax.axvline(np.median(vals), color="k", ls="--", lw=1.5, label=f"median={np.median(vals):.3g}")
    ax.set_xlabel(labels[par], fontsize=10)
    ax.set_ylabel("N events", fontsize=10)
    ax.set_title(f"σ = {np.std(vals):.3g}", fontsize=9)
    ax.legend(fontsize=8)
fig2.suptitle(
    f"SALT2 (free z) parameter distributions — {OBJ_CLASS}  (N={len(df_ok)} events,  {SALT2_SOURCE})",
    fontsize=11,
)
plt.show()

## 9 · Hubble diagram: distance modulus vs redshift

Tripp-formula estimate of the distance modulus:

$$
\mu = m_B^* - M_B + \alpha\, x_1 - \beta\, c
$$

with $\alpha = 0.14$, $\beta = 3.1$ (Betoule et al. 2014).
The apparent magnitude $m_B^*$ is derived from $x_0$ via
$m_B^* = -2.5\log_{10}(x_0) + 10.635$.

Error bars on $\mu$ come from `computemustaterrors` via $\sigma_\mu = \sqrt{\mathbf{g}^T C_{\rm sub}\, \mathbf{g}}$.

In [ ]:
YMIN = 35.0
YMAX = 48.0


ALPHA_TRIPP = 0.14
BETA_TRIPP = 3.10
MB_CONST = 10.635  # salt2-extended normalisation constant
MB_CORR_SDC = +15.0  # empirical offset to centre the Hubble diagram

df_ok = df_ok.copy()
df_ok["mB_star"] = -2.5 * np.log10(df_ok["x0"].astype(float)) + MB_CONST
df_ok["mu_tripp"] = df_ok["mB_star"] + ALPHA_TRIPP * df_ok["x1"] - BETA_TRIPP * df_ok["c"] + MB_CORR_SDC

# ── Plot: Hubble diagram ───────────────────────────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(8, 6), tight_layout=True)
# plot errorbar
ax3.errorbar(
    df_ok["z_true"],
    df_ok["mu_tripp"],
    yerr=df_ok["sigma_mu"],
    lw=0,
    marker="",
    color="grey",
    ecolor="grey",
    capsize=2,
    elinewidth=1,
    alpha=0.2,
)
# plot scattered points
sc = ax3.scatter(
    df_ok["z_true"],
    df_ok["mu_tripp"],
    c=df_ok["chi2_red"],
    cmap="viridis_r",
    vmin=0,
    vmax=5,
    s=40,
    edgecolors="k",
    linewidths=0.3,
    zorder=3,
)
plt.colorbar(sc, ax=ax3, label=r"$\chi^2$/dof")

try:
    from astropy.cosmology import FlatLambdaCDM

    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
    z_ref = np.linspace(max(Z_MIN, 0.01), Z_MAX, 300)
    mu_ref = cosmo.distmod(z_ref).value
    ax3.plot(z_ref, mu_ref, "r--", lw=1.5, label=r"Flat ΛCDM ($H_0$=70, $\Omega_m$=0.3)")
    ax3.legend(fontsize=9)
except Exception:
    pass

ax3.set_xlabel("Redshift z (truth)", fontsize=11)
ax3.set_ylabel(r"$\mu_{\rm Tripp}$ (relative)", fontsize=11)
ax3.set_title(
    f"Hubble diagram — {OBJ_CLASS}  ({len(df_ok)} events)\n"
    rf"Model: {SALT2_SOURCE}  |  "
    rf"$\mu = m_B^* + {ALPHA_TRIPP}\,x_1 - {BETA_TRIPP}\,c$ (fit z)",
    fontsize=11,
)
ax3.set_ylim(YMIN, YMAX)
plt.show()

# ── Hubble residuals ───────────────────────────────────────────────────────────
try:
    df_ok["mu_lcdm"] = cosmo.distmod(df_ok["z_true"].values).value
    df_ok["mu_resid"] = df_ok["mu_tripp"] - df_ok["mu_lcdm"]

    fig4, ax4 = plt.subplots(figsize=(8, 4), tight_layout=True)
    ax4.errorbar(
        df_ok["z_true"],
        df_ok["mu_resid"],
        yerr=df_ok["sigma_mu"],
        lw=0,
        marker="",
        color="grey",
        ecolor="grey",
        capsize=2,
        elinewidth=1,
        alpha=0.2,
    )
    sc4 = ax4.scatter(
        df_ok["z_true"],
        df_ok["mu_resid"],
        c=df_ok["chi2_red"],
        cmap="viridis_r",
        vmin=0,
        vmax=5,
        s=35,
        edgecolors="k",
        linewidths=0.3,
    )
    ax4.axhline(0.0, color="r", ls="--", lw=1.5)
    rms = np.std(df_ok["mu_resid"].dropna())
    ax4.set_xlabel("Redshift z (truth)", fontsize=11)
    ax4.set_ylabel(r"$\Delta\mu = \mu_{\rm Tripp} - \mu_{\Lambda{\rm CDM}}$", fontsize=10)
    ax4.set_title(f"Hubble residuals — {SALT2_SOURCE}  |  RMS = {rms:.3f} mag (fit z)", fontsize=11)
    ax4.set_ylim(-3.5, 3.5)
    ax4.set_xlim(0, Z_MAX)
    plt.colorbar(sc4, ax=ax4, label=r"$\chi^2$/dof")
    plt.show()
    print(f"Hubble residual RMS = {rms:.4f} mag  (N={len(df_ok)} events)")
except Exception as e:
    print(f"Could not compute Hubble residuals: {e}")

## 10 · Photometric redshift performance: `z_fit` vs `z_true`

This is the key diagnostic of free-z SALT2 fitting — identical in structure to the
corresponding section in notebook `03b` for SALT3, enabling a direct comparison.

Three panels:
1. **Main**: errorbar plot `z_fit ± σ_z` vs `z_true`, colour-coded by χ²/dof.
   The dashed diagonal is the ideal $z_{\rm fit} = z_{\rm true}$ line.
2. **Residual**: $\Delta z = z_{\rm fit} - z_{\rm true}$ vs `z_true`.
3. **Normalised residual**: $\Delta z / (1 + z_{\rm true})$ vs `z_true`
   (the standard photo-z figure of merit).

In [ ]:
# The following code aligh the 3 plot vertically exactly

# ── Collect arrays ─────────────────────────────────────────────────────────────
z_true_arr = df_ok["z_true"].values
z_fit_arr = df_ok["z_fit"].values
z_err_arr = df_ok["z_err"].values
chi2_arr = df_ok["chi2_red"].values
dz_arr = df_ok["dz"].values
dz_1pz_arr = df_ok["dz_1pz"].values

# Clamp z_err for display (very large errors distort the figure)
Z_ERR_MAX = 0.3
z_err_display = np.clip(z_err_arr, 0, Z_ERR_MAX)

rms_dz = float(df_ok["dz"].std())
rms_dz1pz = float(df_ok["dz_1pz"].std())

# ── Build figure ───────────────────────────────────────────────────────────────

# --- Création de la grille ---
fig11 = plt.figure(figsize=(8, 14))  # Hauteur pour 3 figures empilées
gs = GridSpec(3, 2, width_ratios=[1, 0.05], height_ratios=[3, 1.4, 1.4], hspace=0.3, wspace=0.1)

cmap = plt.cm.viridis_r
norm = matplotlib.colors.Normalize(vmin=0, vmax=5)
colors = cmap(norm(np.clip(chi2_arr, 0, 5)))


# --- Figure 1 (avec colorbar) ---
ax1 = fig11.add_subplot(gs[0, 0])  # Première ligne, première colonne

for i in range(len(z_true_arr)):
    ax1.errorbar(
        z_true_arr[i],
        z_fit_arr[i],
        yerr=z_err_display[i],
        fmt="o",
        ms=4,
        color=colors[i],
        ecolor=colors[i],
        alpha=0.75,
        capsize=2,
        lw=0.8,
        zorder=3,
    )

z_ref = np.linspace(max(Z_BOUND_LO, z_true_arr.min() - 0.02), min(Z_BOUND_HI, z_true_arr.max() + 0.02), 200)
ax1.plot(z_ref, z_ref, "k--", lw=1.5, label=r"$z_{\rm fit} = z_{\rm true}$", zorder=2)
ax1.fill_between(z_ref, z_ref - rms_dz, z_ref + rms_dz, alpha=0.10, color="k", label=f"±RMS = {rms_dz:.4f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar1 = fig11.colorbar(sm, cax=fig11.add_subplot(gs[0, 1]))  # Colorbar dans la 2ème colonne
cbar1.set_label(r"$\chi^2$/dof", fontsize=10)

ax1.set_ylabel(r"$z_{\rm fit}$ (SALT2, free $z$)", fontsize=11)
ax1.set_xlabel(r"$z_{\rm true}$", fontsize=11)
ax1.set_title(
    f"Photometric redshift from SALT2 light-curve fit — {OBJ_CLASS}\n"
    f"N={len(df_ok)} events  |  "
    f"RMS Δz = {rms_dz:.4f}  |  "
    f"RMS Δz/(1+z) = {rms_dz1pz:.5f}\n"
    f"Model: {SALT2_SOURCE}  |  Error bars capped at {Z_ERR_MAX} for display (fit z)",
    fontsize=10,
)
ax1.legend(fontsize=9, loc="upper left")


# ── Panel 2: Δz = z_fit − z_true ──────────────────────────────────────────────
ax2 = fig11.add_subplot(gs[1, 0])  # Deuxième ligne, première colonne
for i in range(len(z_true_arr)):
    ax2.errorbar(
        z_true_arr[i],
        dz_arr[i],
        yerr=z_err_display[i],
        fmt="o",
        ms=4,
        color=colors[i],
        ecolor=colors[i],
        alpha=0.75,
        capsize=2,
        lw=0.8,
        zorder=3,
    )
ax2.axhline(0.0, color="k", lw=1.2, ls="--")
ax2.axhline(+rms_dz, color="gray", lw=0.8, ls=":")
ax2.axhline(-rms_dz, color="gray", lw=0.8, ls=":")

try:
    z_bins = np.linspace(z_true_arr.min(), z_true_arr.max(), 8)
    z_bin_cen = 0.5 * (z_bins[:-1] + z_bins[1:])
    dz_median = [
        np.median(dz_arr[(z_true_arr >= z_bins[k]) & (z_true_arr < z_bins[k + 1])])
        for k in range(len(z_bin_cen))
    ]
    ax2.plot(z_bin_cen, dz_median, "rs-", ms=6, lw=1.5, label="binned median Δz", zorder=5)
    ax2.legend(fontsize=8)
except Exception:
    pass

ax2.set_ylabel(r"$\Delta z = z_{\rm fit} - z_{\rm true}$", fontsize=11)
ax2.set_title(f"Redshift residuals  |  RMS = {rms_dz:.4f}", fontsize=9)
ax2.set_xlabel(r"$z_{\rm true}$", fontsize=11)


# ── Panel 3: Δz / (1+z_true) ──────────────────────────────────────────────────
ax3 = fig11.add_subplot(gs[2, 0])  # Troisième ligne, première colonne

for i in range(len(z_true_arr)):
    ax3.errorbar(
        z_true_arr[i],
        dz_1pz_arr[i],
        yerr=z_err_display[i] / (1.0 + z_true_arr[i]),
        fmt="o",
        ms=4,
        color=colors[i],
        ecolor=colors[i],
        alpha=0.75,
        capsize=2,
        lw=0.8,
        zorder=3,
    )
ax3.axhline(0.0, color="k", lw=1.2, ls="--")
ax3.axhline(+rms_dz1pz, color="gray", lw=0.8, ls=":", label=f"±RMS={rms_dz1pz:.5f}")
ax3.axhline(-rms_dz1pz, color="gray", lw=0.8, ls=":")

try:
    dz1pz_median = [
        np.median(dz_1pz_arr[(z_true_arr >= z_bins[k]) & (z_true_arr < z_bins[k + 1])])
        for k in range(len(z_bin_cen))
    ]
    ax3.plot(z_bin_cen, dz1pz_median, "rs-", ms=6, lw=1.5, label="binned median", zorder=5)
    ax3.legend(fontsize=8)
except Exception:
    pass

ax3.set_xlabel(r"$z_{\rm true}$", fontsize=11)
ax3.set_ylabel(r"$\Delta z\,/\,(1+z_{\rm true})$", fontsize=11)
ax3.set_title(rf"Normalised residuals  |  RMS = {rms_dz1pz:.5f}  ", fontsize=9)


# plt.tight_layout()
plt.show()

## 11 · Distribution of redshift residuals

In [ ]:
fig12, axes12 = plt.subplots(1, 3, figsize=(14, 4), tight_layout=True)

# Panel A: histogram of z_err
vals_zerr = df_ok["z_err"].clip(upper=Z_ERR_MAX)
axes12[0].hist(vals_zerr, bins=20, color="C0", edgecolor="white")
axes12[0].axvline(
    np.median(vals_zerr), color="k", ls="--", lw=1.5, label=f"median={np.median(vals_zerr):.4f}"
)
axes12[0].set_xlabel(r"$\sigma_z$ from fit (capped at 0.3)", fontsize=10)
axes12[0].set_ylabel("N events", fontsize=10)
axes12[0].set_title(f"σ(z_err) = {vals_zerr.std():.4f}", fontsize=9)
axes12[0].legend(fontsize=8)

# Panel B: histogram of Δz
axes12[1].hist(df_ok["dz"], bins=25, color="C1", edgecolor="white")
axes12[1].axvline(
    np.median(df_ok["dz"]), color="k", ls="--", lw=1.5, label=f"median={np.median(df_ok['dz']):+.4f}"
)
axes12[1].axvline(0.0, color="r", ls=":", lw=1.0)
axes12[1].set_xlabel(r"$\Delta z = z_{\rm fit} - z_{\rm true}$", fontsize=10)
axes12[1].set_ylabel("N events", fontsize=10)
axes12[1].set_title(f"RMS = {df_ok['dz'].std():.4f}", fontsize=9)
axes12[1].legend(fontsize=8)

# Panel C: histogram of Δz/(1+z)
axes12[2].hist(df_ok["dz_1pz"], bins=25, color="C2", edgecolor="white")
axes12[2].axvline(
    np.median(df_ok["dz_1pz"]), color="k", ls="--", lw=1.5, label=f"median={np.median(df_ok['dz_1pz']):+.5f}"
)
axes12[2].axvline(0.0, color="r", ls=":", lw=1.0)
axes12[2].set_xlabel(r"$\Delta z\,/\,(1+z_{\rm true})$", fontsize=10)
axes12[2].set_ylabel("N events", fontsize=10)
axes12[2].set_title(f"RMS = {df_ok['dz_1pz'].std():.5f}", fontsize=9)
axes12[2].legend(fontsize=8)

fig12.suptitle(
    f"Redshift recovery — SALT2 free-z fit — {OBJ_CLASS}  (N={len(df_ok)} events,  {SALT2_SOURCE})",
    fontsize=11,
)
plt.show()

## Summary

### Parameter status in this notebook

| Parameter | Notebook 02 (z fixed) | This notebook 02b (z free) |
|-----------|----------------------|-----------------------------|
| `z`  | **fixed** to truth ZCMB | **free** — fitted from light curves |
| `t0` | free | free |
| `x0` | free | free |
| `x1` | free | free |
| `c`  | free | free |

### Two-pass fitting strategy

| Pass | Parameters | Purpose |
|------|------------|--------|
| 1 — coarse grid scan | `t0, x0, x1, c` (z fixed on grid) | Locate the global χ² minimum in z |
| 2 — refined free fit | `z, t0, x0, x1, c` (all free) | Precise parameters + covariance |

### SALT2 vs SALT3 comparison

Run notebook `03b_elasticc2_fitsalt3andredshift_lightcurves.ipynb` with the same
`N_CURVES` and `RANDOM_SEED` to compare the photometric redshift accuracy of SALT2
(`salt2-extended`) and SALT3 (`salt3` v2.0) on identical ELAsTiCC2 samples.
The key figure of merit is RMS Δz/(1+z) from section 10.

### Key caveats

- Without spectroscopic host redshift, the z–c degeneracy limits precision,
  especially at high redshift where rest-frame UV features shift out of the LSST
  optical window.
- The SALT2-extended template covers rest-frame 2000–9200 Å;
  the SALT3 template covers 2000–11000 Å — SALT3 has slightly better coverage
  at high redshift in the y-band.
- The σ_z from `sncosmo.fit_lc` is the **statistical** uncertainty from the
  covariance matrix and underestimates the true uncertainty.
- Events with χ²/dof >> 1 should be filtered before cosmological analysis.
